# Exploratory Data Analysis: Synthetic Support Tickets

This notebook loads, inspects, and analyzes the synthetic customer support dataset generated by Gemini Batch (`data/generated/run_v1/results.jsonl`).

### Objectives:
1. **Unpack & Flatten**: Extract nested batch responses into a flat, tabular Pandas DataFrame.
2. **Audit Integrity**: Inspect missing values, data types, and schema completeness.
3. **Analyze Distributions**: Examine interaction types, categories, actions, and escalation reasons.
4. **Categorical analysis**: Examine the spread of categories in the dataset.
5. **Text Length Profiling**: Spot empty responses, truncations, or outliers.
6. **Cleaning Pipeline**: Test cleaning and filtering rules.

In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

# Set Pandas display options for readable tables in notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# Determine project root and data path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "generated" / "run_v1" / "results.jsonl"
print(f"Data file: {DATA_FILE}")
print(f"Exists: {DATA_FILE.exists()}")

Data file: /home/jucruz/MySchoolWork/year3/FallQuarter/PRO390/data/generated/run_v1/results.jsonl
Exists: True


## 1. Load and Unroll Synthetic Support Tickets

Each line in `results.jsonl` contains a Gemini Batch candidate with a stringified JSON payload in `tickets`. We parse the inner JSON and flatten all tickets into individual DataFrame rows.

In [3]:
def extract_tickets_from_raw_text(text: str) -> list[dict]:
    """Clean markdown code fencing if present and parse the inner JSON string."""
    cleaned = text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    data = json.loads(cleaned)
    if isinstance(data, dict):
        return data.get("tickets", [])
    elif isinstance(data, list):
        return data
    return []

def load_tickets_dataframe(file_path: Path) -> pd.DataFrame:
    """Read results.jsonl and unroll all tickets into a Pandas DataFrame."""
    records = []
    errors = 0
    line_count = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f, start=1):
            line_str = line.strip()
            if not line_str:
                continue
            line_count = line_idx
            try:
                raw = json.loads(line_str)
                batch_key = raw.get("key", f"line_{line_idx}")
                candidates = raw.get("response", {}).get("candidates", [])
                if not candidates:
                    errors += 1
                    continue
                parts = candidates[0].get("content", {}).get("parts", [])
                if not parts or "text" not in parts[0]:
                    errors += 1
                    continue

                tickets = extract_tickets_from_raw_text(parts[0]["text"])
                for t_idx, ticket in enumerate(tickets):
                    msgs = ticket.get("messages", [])
                    customer_msgs = [m.get("content", "") for m in msgs if m.get("role") == "customer"]
                    latest_query = customer_msgs[-1] if customer_msgs else ""

                    records.append({
                        "ticket_id": f"{batch_key}_t{t_idx}",
                        "batch_key": batch_key,
                        "interaction_type": ticket.get("interaction_type"),
                        "category": ticket.get("category"),
                        "action": ticket.get("action"),
                        "is_answerable": ticket.get("is_answerable"),
                        "escalation_reason": ticket.get("escalation_reason"),
                        "confidence": ticket.get("confidence"),
                        "customer_query": latest_query,
                        "answer": ticket.get("answer"),
                        "rationale": ticket.get("rationale"),
                        "ticket_summary": ticket.get("ticket_summary"),
                        "num_messages": len(msgs),
                        "messages": msgs,
                        "evidence": ticket.get("evidence", []),
                    })
            except Exception as e:
                errors += 1

    print(f"Processed {line_count} lines. Parse errors: {errors}")
    return pd.DataFrame(records)

df = load_tickets_dataframe(DATA_FILE)
print(f"Loaded DataFrame with shape: {df.shape}")

Processed 1670 lines. Parse errors: 0
Loaded DataFrame with shape: (5000, 15)


## 2. Tabular Data Preview

Preview the first 5 records in a formatted HTML table.

In [4]:
preview_cols = [
    "ticket_id", "interaction_type", "category", "action", 
    "is_answerable", "confidence", "customer_query", "answer"
]
df[preview_cols].head(5)

,ticket_id,interaction_type,category,action,is_answerable,confidence,customer_query,answer
0,cinder_and_salt-0000_t0,single_qa,catering_and_events,answer,True,0.95,"Hi, planning an outdoor afternoon reception where temps might hit the mid-70s. How long can a bu...","If temperatures are above 72°F (22°C), outdoor display is strictly limited to a maximum of 2 hou..."
1,cinder_and_salt-0000_t1,unanswerable,wholesale_accounts,escalate,False,0.20,Hello there! I'm opening a small coffee shop in Zone A next month and looking into setting up a ...,I don't have information on whether we provide display crates or retail shelving fixtures for wh...
2,cinder_and_salt-0000_t2,escalation,allergen_reaction,escalate,True,0.95,"Hey team, my kid just broke out in hives after having a slice of the sourdough bread we bought t...","Please seek medical attention immediately, and call 911 if there is any difficulty breathing, sw..."
3,cinder_and_salt-0001_t0,unanswerable,bread_storage,escalate,False,0.20,"Hi there, quick question between meetings—I bought your Country Sourdough yesterday and wanted t...","I cannot confirm specific air fryer settings or instructions for refreshing our loaves, as our g..."
4,cinder_and_salt-0001_t1,single_qa,pastry_schedule,answer,True,0.95,"Hello dear, my grandson told me about your cinnamon buns made with rye flour. Could you tell me ...","Our Rye Cinnamon Morning Buns are baked on Saturdays and Sundays, and they are $5.25 each."


## 3. Data Integrity & Missing Values Audit

In [5]:
audit_df = pd.DataFrame({
    "Data Type": df.dtypes,
    "Non-Null Count": df.notnull().sum(),
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2)
})
audit_df

,Data Type,Non-Null Count,Missing Count,Missing %
ticket_id,str,5000,0,0.0
batch_key,str,5000,0,0.0
interaction_type,str,5000,0,0.0
category,str,5000,0,0.0
action,str,5000,0,0.0
is_answerable,bool,5000,0,0.0
escalation_reason,str,5000,0,0.0
confidence,float64,5000,0,0.0
customer_query,str,5000,0,0.0
answer,str,5000,0,0.0


## 4. Categorical Distributions

Check the distribution across interaction types, actions, and categories.

In [6]:
# Distribution by Interaction Type
itype_df = df["interaction_type"].value_counts().reset_index()
itype_df.columns = ["Interaction Type", "Count"]
itype_df["Share (%)"] = (itype_df["Count"] / len(df) * 100).round(2)
itype_df

,Interaction Type,Count,Share (%)
0,single_qa,2250,45.0
1,unanswerable,1000,20.0
2,multi_turn,750,15.0
3,escalation,500,10.0
4,inference,500,10.0


In [7]:
# Distribution by Action
action_df = df["action"].value_counts().reset_index()
action_df.columns = ["Action", "Count"]
action_df["Share (%)"] = (action_df["Count"] / len(df) * 100).round(2)
action_df

,Action,Count,Share (%)
0,answer,3499,69.98
1,escalate,1501,30.02


In [8]:
# Top 15 Categories
cat_df = df["category"].value_counts().head(15).reset_index()
cat_df.columns = ["Category", "Count"]
cat_df["Share (%)"] = (cat_df["Count"] / len(df) * 100).round(2)
cat_df

,Category,Count,Share (%)
0,catering_and_events,189,3.78
1,post_op_care,178,3.56
2,wholesale_accounts,147,2.94
3,sizing_and_fit,145,2.90
4,bread_storage,140,2.80
5,allergens,140,2.80
6,fleet_services,133,2.66
7,chemical_safety,116,2.32
8,returns_and_exchanges,98,1.96
9,warranty_and_repair,93,1.86


## 5. Text Length & Quality Profiling

Analyze text length statistics across queries and answers to identify empty strings, excessive lengths, or outliers.

In [9]:
length_profile = pd.DataFrame({
    "Query Char Length": df["customer_query"].astype(str).str.len(),
    "Query Word Count": df["customer_query"].astype(str).apply(lambda s: len(s.split())),
    "Answer Char Length": df["answer"].astype(str).str.len(),
    "Answer Word Count": df["answer"].astype(str).apply(lambda s: len(s.split())),
    "Confidence Score": df["confidence"]
})
length_profile.describe().round(2)

,Query Char Length,Query Word Count,Answer Char Length,Answer Word Count,Confidence Score
count,5000.00,5000.00,5000.00,5000.00,5000.00
mean,214.85,39.20,274.52,45.76,0.79
std,128.71,22.79,104.95,17.20,0.32
min,19.00,3.00,43.00,7.00,0.00
25%,99.00,18.00,208.00,35.00,0.90
50%,193.00,36.00,265.00,45.00,0.95
75%,307.25,55.00,331.00,55.00,0.95
max,792.00,130.00,962.00,156.00,1.00


## 6. Escalation & Unanswerable Inspection

Inspect the reasons why tickets were marked for escalation or unanswerable.

In [10]:
escalations = df[df["action"] == "escalate"]
esc_summary = escalations["escalation_reason"].value_counts().reset_index()
esc_summary.columns = ["Escalation Reason", "Count"]
esc_summary["Share (%)"] = (esc_summary["Count"] / len(escalations) * 100).round(2)
esc_summary

,Escalation Reason,Count,Share (%)
0,not_in_docs,1000,66.62
1,docs_require_human,501,33.38


## 7. Data cleaning

Testing out different data cleaning ideas

In [ ]:
def clean_dataset(df_raw: pd.DataFrame) -> pd.DataFrame:
    cleaned = df_raw.copy()

    text_cols = ["customer_query", "answer", "rationale", "ticket_summary", "category"]
    for col in text_cols:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].astype(str).str.strip()

    # 2. Filter out any tickets with blank queries or answers
    cleaned = cleaned[cleaned["customer_query"].str.len() > 0]
    cleaned = cleaned[cleaned["answer"].str.len() > 0]

    # 3. Deduplicate queries
    cleaned = cleaned.drop_duplicates(subset=["customer_query"])

    return cleaned

df_cleaned = clean_dataset(df)
print(f"Original rows: {len(df):,} | Cleaned rows: {len(df_cleaned):,}")

# Preview cleaned dataset
df_cleaned[preview_cols].head(3)

Original rows: 5,000 | Cleaned rows: 4,994


,ticket_id,interaction_type,category,action,is_answerable,confidence,customer_query,answer
0,cinder_and_salt-0000_t0,single_qa,catering_and_events,answer,True,0.95,"Hi, planning an outdoor afternoon reception where temps might hit the mid-70s. How long can a bu...","If temperatures are above 72°F (22°C), outdoor display is strictly limited to a maximum of 2 hou..."
1,cinder_and_salt-0000_t1,unanswerable,wholesale_accounts,escalate,False,0.20,Hello there! I'm opening a small coffee shop in Zone A next month and looking into setting up a ...,I don't have information on whether we provide display crates or retail shelving fixtures for wh...
2,cinder_and_salt-0000_t2,escalation,allergen_reaction,escalate,True,0.95,"Hey team, my kid just broke out in hives after having a slice of the sourdough bread we bought t...","Please seek medical attention immediately, and call 911 if there is any difficulty breathing, sw..."
